# Visualizzazione Interattiva Componenti Salvati

Questo notebook carica componenti (embedding) e label di clustering già calcolati e salvati in precedenza, permettendo una ispezione visiva approfondita.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline

In [ ]:
# Caricamento dati salvati
embedding_path = Path('../results/lesion/dim_reduction/tsne/23-07_s1.1_p60/matrix.npy')
cluster_meta_path = Path('../results/lesion/dim_reduction_clustering/tsne/kmeans/26-07_s1.1_p30_k5/metadata.csv')
original_matrix_path = Path('../data/derived/lesion_matrix/21-07_s1.1/matrix.npy')
clinical_meta_path = Path('../assets/metadata/UNIPD_WashU_participants_lesions.tsv')

print("Caricamento in corso...")
embedding = np.load(embedding_path)
cluster_meta = pd.read_csv(cluster_meta_path)
X_orig = np.load(original_matrix_path)

# Calcoliamo il volume approssimato (in voxel)
cluster_meta['volume_voxel'] = X_orig.sum(axis=1)

# Arricchiamo con i metadati clinici
clinical_meta = pd.read_csv(clinical_meta_path, sep='\t')
cluster_meta = cluster_meta.merge(clinical_meta, left_on='subject_id', right_on='participant_id', how='left')

print(f"Shape embedding: {embedding.shape}")
print(f"Metadata caricati e arricchiti: {cluster_meta.shape}")

In [ ]:
# 1. Colorato per Cluster
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=embedding[:, 0], 
    y=embedding[:, 1], 
    hue=cluster_meta['cluster_label'], 
    palette='tab10', 
    s=25, 
    alpha=0.9
)
plt.title('t-SNE - Colorato per Cluster (K-Means)')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Colorato per Dataset
plt.figure(figsize=(10, 8))
sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], hue=cluster_meta['dataset_x'], palette='Set2', s=25, alpha=0.7)
plt.title('t-SNE - Dataset (Batch Effect Check)')
plt.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Colorato per Lato Lesione
plt.figure(figsize=(10, 8))
sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], hue=cluster_meta['lesion_side'], palette='Set1', s=25, alpha=0.7)
plt.title('t-SNE - Lato Lesione')
plt.legend(title='Lato', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 4. Colorato per Volume Lesionale
plt.figure(figsize=(10, 8))
scatter = plt.scatter(x=embedding[:, 0], y=embedding[:, 1], c=cluster_meta['volume_voxel'], cmap='magma', s=25, alpha=0.8)
plt.title('t-SNE - Volume Lesionale (Voxel)')
plt.colorbar(scatter, label='Numero di Voxel Lesionati')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Colorato per NIHSS (Severity)
plt.figure(figsize=(10, 8))
scatter = plt.scatter(x=embedding[:, 0], y=embedding[:, 1], c=cluster_meta['NIHSS'], cmap='viridis', s=25, alpha=0.8)
plt.title('t-SNE - Severità NIHSS')
plt.colorbar(scatter, label='NIHSS Score')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Kernel Density Estimation (Densità)
plt.figure(figsize=(10, 8))
sns.kdeplot(
    x=embedding[:, 0], 
    y=embedding[:, 1], 
    cmap="Blues", 
    fill=True, 
    thresh=0.05, 
    alpha=0.5
)
sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], color='black', s=5, alpha=0.3)
plt.title('t-SNE - Densità (KDE) dell\'embedding')
plt.tight_layout()
plt.show()